# 06 — Double DQN, Dueling DQN, and Prioritized Replay in JAX

**CartPole-v1 · JAX + Flax + Optax + Gymnax**

This notebook extends Notebook 05 with three independently switchable improvements:

- **Double DQN:** select the next action using the online network, evaluate it using the target network.
- **Dueling architecture:** represent Q-values as a state value and centered action advantages.
- **Prioritized experience replay (PER):** sample transitions according to TD-error magnitude, with importance-sampling correction.

Includes unit tests, end-to-end training, evaluation, factorial ablations, and optional multi-seed experiments. The default run trains standard DQN and the combined improved agent. The larger ablations are opt-in.

**Methodological caution:** prioritized replay samples nonuniformly; importance weights correct its sampling distribution in expectation when $\beta=1$, but not the broader off-policy bootstrapping bias.


In [1]:
# In a fresh environment, uncomment, run, and restart the kernel if needed:
# %pip install -q "jax>=0.4.30" "flax>=0.10" "optax>=0.2.3" "gymnax>=0.0.8" matplotlib pandas



## 1. Mathematical Definitions

### 1.1 Standard DQN

Deep Q-Networks (DQN) approximate the optimal action-value function using a neural network $Q_\theta(s,a)$.

For a transition $(s,a,r,s',d)$, the Bellman target is

$$
y^{\mathrm{DQN}} = r + \gamma(1-d)\max_{a'} Q_{\theta^-}(s',a'),
$$

where $\theta$ and $\theta^-$ denote the online and target network parameters, respectively, and $d$ indicates episode termination.

The temporal-difference (TD) error is

$$
\delta = y^{\mathrm{DQN}} - Q_\theta(s,a).
$$

The target network is updated periodically to stabilize learning.

### 1.2 Double DQN

Standard DQN uses the same target network to select and evaluate the maximizing next action. This can lead to overestimation bias.

**Double DQN** separates action selection from action evaluation:

- The online network selects the maximizing action.
- The target network evaluates the selected action.

The Double DQN target is

$$
y^{\mathrm{DDQN}}
=
r+\gamma(1-d)
Q_{\theta^-}\left(s',\arg\max_{a'}Q_\theta(s',a')\right).
$$

This decoupling can reduce overestimation bias.

### 1.3 Dueling DQN

Instead of directly predicting $Q_\theta(s,a)$, the dueling architecture learns two quantities:

- **State value:** $V_\theta(s)$ measures the desirability of a state.
- **Action advantage:** $A_\theta(s,a)$ measures the relative desirability of an action.

They are combined as

$$
Q_\theta(s,a)
=
V_\theta(s)
+
A_\theta(s,a)
-
\frac{1}{|\mathcal A|}
\sum_{b\in\mathcal A} A_\theta(s,b).
$$

Subtracting the mean advantage resolves the ambiguity in the value-advantage decomposition and ensures that

$$
\frac{1}{|\mathcal A|}\sum_{a\in\mathcal A}Q_\theta(s,a)=V_\theta(s).
$$

This architecture can be useful when several actions have similar values.

### 1.4 Prioritized Experience Replay (PER)

Standard experience replay samples transitions uniformly. **Prioritized Experience Replay** assigns higher sampling probabilities to transitions with larger TD errors.

For transition $i$, define the priority

$$
p_i = (|\delta_i|+\epsilon_{\mathrm{PER}})^\alpha,
$$

where $\epsilon_{\mathrm{PER}}>0$ prevents zero priorities and $\alpha\geq 0$ controls prioritization strength.

The sampling probability is

$$
P(i)=\frac{p_i}{\sum_{j=1}^{N}p_j},
$$

where $N$ is the number of transitions currently stored in the replay buffer.

- $\alpha=0$ corresponds to uniform sampling.
- Larger $\alpha$ emphasizes transitions with larger TD errors.

Because nonuniform sampling introduces bias relative to the uniform replay objective, PER uses importance-sampling weights:

$$
w_i =
\frac{(NP(i))^{-\beta}}
{\max_{1\leq j\leq N}(NP(j))^{-\beta}},
$$

where $\beta\in[0,1]$ controls the correction strength.

When $\beta=1$, the weights correct the sampling distribution for the empirical replay objective, up to a common normalization factor.

### 1.5 Importance-Weighted TD Loss

For a minibatch $\mathcal B$ containing $B$ transitions, we minimize

$$
\mathcal L(\theta)
=
\frac{1}{B}
\sum_{i\in\mathcal B}
w_i\,\ell_{\mathrm{Huber}}
\left(Q_\theta(s_i,a_i)-y_i\right),
$$

where $y_i$ is computed using either the standard DQN or Double DQN target.

The Huber loss is

$$
\ell_{\mathrm{Huber}}(e)
=
\begin{cases}
\frac12 e^2, & |e|\leq 1,\\
|e|-\frac12, & |e|>1.
\end{cases}
$$

After each optimization step, sampled transitions' priorities are refreshed using the absolute TD errors computed **before** the parameter update. Newly collected transitions receive the largest priority observed so far.

### 1.6 Summary

| Method | Modification | Purpose |
|---|---|---|
| Double DQN | Bellman target | Reduce Q-value overestimation |
| Dueling DQN | Network architecture | Separate state value and action advantage |
| PER | Replay sampling and loss weighting | Emphasize high-TD-error transitions |

These three improvements address different aspects of DQN and can be combined into a single algorithm.


In [2]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import flax.linen as nn
import optax, gymnax
from dataclasses import dataclass, replace
from functools import partial
from typing import NamedTuple

env, env_params = gymnax.make("CartPole-v1")
sample_obs, _ = env.reset(jax.random.PRNGKey(0), env_params)
OBS_DIM = int(sample_obs.shape[-1])
N_ACTIONS = int(env.num_actions)
print("JAX:", jax.__version__, "devices:", jax.devices(),
      "observation:", OBS_DIM, "actions:", N_ACTIONS)


JAX: 0.11.1 devices: [CudaDevice(id=0), CudaDevice(id=1)] observation: 4 actions: 2


## 2. Configurable Q-networks

In [3]:
@dataclass(frozen=True)
class Config:
    seed: int = 0
    total_steps: int = 100_000
    learning_starts: int = 1_000
    train_frequency: int = 4
    target_update_frequency: int = 500
    buffer_capacity: int = 20_000
    batch_size: int = 64
    gamma: float = 0.99
    lr: float = 1e-3
    hidden: int = 64
    grad_clip: float = 10.
    epsilon_start: float = 1.
    epsilon_end: float = .05
    epsilon_decay_steps: int = 35_000
    eval_every: int = 5_000
    eval_episodes: int = 30
    max_episode_steps: int = 500
    double: bool = True
    dueling: bool = True
    prioritized: bool = True
    per_alpha: float = .6
    per_beta_start: float = .4
    per_beta_steps: int = 100_000
    per_eps: float = 1e-5

CFG = Config()

class QNet(nn.Module):
    actions: int
    hidden: int
    dueling: bool
    @nn.compact
    def __call__(self, obs):
        x = nn.relu(nn.Dense(self.hidden)(obs))
        x = nn.relu(nn.Dense(self.hidden)(x))
        if self.dueling:
            value = nn.Dense(1, name="value")(x)
            advantage = nn.Dense(self.actions, name="advantage")(x)
            return value + advantage - jnp.mean(advantage, axis=-1, keepdims=True)
        return nn.Dense(self.actions, name="q_head")(x)

def epsilon(step, cfg):
    f = min(step / max(cfg.epsilon_decay_steps, 1), 1.)
    return cfg.epsilon_start + f * (cfg.epsilon_end - cfg.epsilon_start)

def beta(step, cfg):
    f = min(step / max(cfg.per_beta_steps, 1), 1.)
    return cfg.per_beta_start + f * (1. - cfg.per_beta_start)

def env_step(key, state, action):
    outputs = env.step(key, state, action, env_params)
    return outputs[0], outputs[1], outputs[2], outputs[3]

def make_agent(cfg):
    net = QNet(N_ACTIONS, cfg.hidden, cfg.dueling)
    optimizer = optax.chain(optax.clip_by_global_norm(cfg.grad_clip), optax.adam(cfg.lr))
    return net, optimizer


## 3. Replay buffer with priorities

For clarity, sampling computes a probability vector over the full fixed-capacity array and masks unused slots. This is easy to verify but costs $O(C)$ per PER sample. Large-scale implementations should use a sum tree for $O(\\log C)$ sampling and updates.

We sample **with replacement**. Repeated indices may occur, so priority updates are aggregated using the maximum TD error for each sampled index before writing back.


In [4]:
class Buffer(NamedTuple):
    obs: jax.Array
    action: jax.Array
    reward: jax.Array
    next_obs: jax.Array
    done: jax.Array
    priority: jax.Array
    pos: jax.Array
    size: jax.Array
    max_priority: jax.Array

def init_buffer(capacity):
    return Buffer(
        jnp.zeros((capacity, OBS_DIM), jnp.float32),
        jnp.zeros((capacity,), jnp.int32),
        jnp.zeros((capacity,), jnp.float32),
        jnp.zeros((capacity, OBS_DIM), jnp.float32),
        jnp.zeros((capacity,), jnp.bool_),
        jnp.zeros((capacity,), jnp.float32),
        jnp.array(0, jnp.int32),
        jnp.array(0, jnp.int32),
        jnp.array(1., jnp.float32),
    )

@jax.jit
def add(buffer, obs, action, reward, next_obs, done):
    i = buffer.pos
    capacity = buffer.action.shape[0]
    return Buffer(
        buffer.obs.at[i].set(obs),
        buffer.action.at[i].set(action),
        buffer.reward.at[i].set(reward),
        buffer.next_obs.at[i].set(next_obs),
        buffer.done.at[i].set(done),
        buffer.priority.at[i].set(buffer.max_priority),
        (i + 1) % capacity,
        jnp.minimum(buffer.size + 1, capacity),
        buffer.max_priority,
    )

@partial(jax.jit, static_argnames=("batch_size", "prioritized"))
def sample(buffer, key, batch_size, prioritized, alpha, beta_value):
    capacity = buffer.action.shape[0]
    valid = jnp.arange(capacity) < buffer.size
    if prioritized:
        weights = jnp.where(valid, jnp.maximum(buffer.priority, 1e-8) ** alpha, 0.)
        probabilities = weights / jnp.sum(weights)
        indices = jax.random.choice(key, capacity, (batch_size,), replace=True, p=probabilities)
        # Normalization by the largest possible IS weight over the populated buffer.
        min_p = jnp.min(jnp.where(valid, probabilities, jnp.inf))
        is_weights = (probabilities[indices] / min_p) ** (-beta_value)
    else:
        indices = jax.random.randint(key, (batch_size,), 0, buffer.size)
        is_weights = jnp.ones((batch_size,), jnp.float32)
    batch = (buffer.obs[indices], buffer.action[indices], buffer.reward[indices],
             buffer.next_obs[indices], buffer.done[indices])
    return indices, batch, is_weights

@jax.jit
def update_priorities(buffer, indices, errors, per_eps):
    # Max over duplicate sampled indices; unsampled positions are unchanged.
    new_p = jnp.zeros_like(buffer.priority).at[indices].max(
        jnp.abs(errors) + per_eps
    )
    touched = jnp.zeros_like(buffer.priority, dtype=jnp.bool_).at[indices].set(True)
    priorities = jnp.where(touched, new_p, buffer.priority)
    return buffer._replace(
        priority=priorities,
        max_priority=jnp.maximum(buffer.max_priority, jnp.max(new_p))
    )

# Circular overwrite test.
b = init_buffer(3)
for i in range(5):
    s = jnp.full((OBS_DIM,), i, jnp.float32)
    b = add(b, s, jnp.int32(i % N_ACTIONS), jnp.float32(i), s + 1, jnp.bool_(False))
assert int(b.size) == 3 and int(b.pos) == 2
assert set(np.asarray(b.reward).tolist()) == {2., 3., 4.}
idx, batch, weights = sample(b, jax.random.PRNGKey(7), 8, True, .6, .4)
assert batch[0].shape == (8, OBS_DIM)
assert np.all(np.asarray(weights) > 0) and np.all(np.asarray(weights) <= 1.00001)
print("Replay and importance-weight tests passed.")


Replay and importance-weight tests passed.


## 4. Double DQN targets and weighted optimizer update

In [5]:
def bellman_target(reward, done, online_next_q, target_next_q, gamma, double):
    if double:
        chosen = jnp.argmax(online_next_q, axis=-1)
        next_value = jnp.take_along_axis(target_next_q, chosen[:, None], axis=1)[:, 0]
    else:
        next_value = jnp.max(target_next_q, axis=-1)
    return reward + gamma * (1. - done.astype(jnp.float32)) * next_value

def make_update(cfg, net, optimizer):
    @jax.jit
    def update(params, target, opt_state, batch, importance):
        obs, action, reward, next_obs, done = batch
        online_next = net.apply(params, next_obs)
        target_next = net.apply(target, next_obs)
        targets = jax.lax.stop_gradient(
            bellman_target(reward, done, online_next, target_next,
                           cfg.gamma, cfg.double)
        )
        def loss_fn(p):
            q = net.apply(p, obs)
            predicted = jnp.take_along_axis(q, action[:, None], axis=1)[:, 0]
            td = targets - predicted
            loss = jnp.mean(importance * optax.huber_loss(td, delta=1.))
            return loss, (td, predicted.mean(), targets.mean())
        (loss, (td, q_mean, target_mean)), grads = jax.value_and_grad(
            loss_fn, has_aux=True
        )(params)
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, (loss, jnp.mean(jnp.abs(td)), q_mean, target_mean), td
    return update

# Double DQN must select using online, but evaluate using target.
on = jnp.array([[10., 2.]])
tar = jnp.array([[1., 9.]])
assert np.allclose(bellman_target(jnp.array([0.]), jnp.array([False]),
                                  on, tar, 1., True), [1.])
assert np.allclose(bellman_target(jnp.array([0.]), jnp.array([False]),
                                  on, tar, 1., False), [9.])
assert np.allclose(bellman_target(jnp.array([2.]), jnp.array([True]),
                                  on, tar, .99, True), [2.])
print("Double DQN and terminal target tests passed.")


Double DQN and terminal target tests passed.


## 5. Greedy and epsilon-greedy evaluation

In [6]:
def select_action(params, obs, key, epsilon_value, net):
    random_key, coin_key = jax.random.split(key)
    greedy = jnp.argmax(net.apply(params, obs))
    random_action = jax.random.randint(random_key, (), 0, N_ACTIONS)
    return jnp.where(jax.random.uniform(coin_key) < epsilon_value,
                     random_action, greedy).astype(jnp.int32)

def make_evaluator(net, max_steps):
    def episode(params, key, greedy):
        rng, reset_key = jax.random.split(key)
        obs, state = env.reset(reset_key, env_params)
        def step(carry, _):
            obs, state, rng, finished = carry
            rng, ak, sk = jax.random.split(rng, 3)
            eps = jnp.where(greedy, 0., .05)
            action = select_action(params, obs, ak, eps, net)
            next_obs, next_state, reward, done = env_step(sk, state, action)
            reward = jnp.where(finished, 0., reward)
            next_obs = jnp.where(finished, obs, next_obs)
            next_state = jax.tree_util.tree_map(
                lambda old, new: jnp.where(finished, old, new),
                state, next_state
            )
            return (next_obs, next_state, rng, finished | done), reward
        _, rewards = jax.lax.scan(
            step, (obs, state, rng, jnp.bool_(False)), None, length=max_steps
        )
        return rewards.sum()
    @jax.jit
    def evaluate(params, keys, greedy):
        return jax.vmap(lambda k: episode(params, k, greedy))(keys)
    return evaluate

def bootstrap_ci(scores, seed=0, draws=2000):
    x = np.asarray(scores)
    rng = np.random.default_rng(seed)
    means = rng.choice(x, (draws, len(x)), replace=True).mean(axis=1)
    return np.quantile(means, [.025, .975])


## 6. End-to-end training

This reference version uses a Python environment-interaction loop with JIT-compiled sampling and gradient updates. The PER probability vector is recomputed every update, which is suitable for CartPole but not optimized for million-entry buffers.

Target synchronization happens at fixed **environment-step** intervals, not optimizer-update intervals. Epsilon and PER beta are annealed by environment steps. We log completed training episodes separately from evaluation returns.


In [7]:
def train(cfg=CFG, seed=None, verbose=True):
    seed = cfg.seed if seed is None else seed
    rng = jax.random.PRNGKey(seed)
    rng, init_key, reset_key = jax.random.split(rng, 3)
    net, optimizer = make_agent(cfg)
    params = net.init(init_key, jnp.zeros((OBS_DIM,), jnp.float32))
    target = params
    opt_state = optimizer.init(params)
    update_fn = make_update(cfg, net, optimizer)
    @jax.jit
    def act(p, observation, key, eps):
        return select_action(p, observation, key, eps, net)
    evaluator = make_evaluator(net, cfg.max_episode_steps)
    eval_keys = jax.random.split(jax.random.PRNGKey(2026), cfg.eval_episodes)
    buffer = init_buffer(cfg.buffer_capacity)
    obs, state = env.reset(reset_key, env_params)
    episode_return, episode_length = 0., 0
    episodes, updates, evaluations = [], [], []
    for step in range(1, cfg.total_steps + 1):
        rng, action_key, step_key, sample_key, reset_key = jax.random.split(rng, 5)
        eps = epsilon(step - 1, cfg)
        action = act(params, obs, action_key, jnp.float32(eps))
        next_obs, next_state, reward, done = env_step(step_key, state, action)
        buffer = add(buffer, obs, action, reward, next_obs, done)
        episode_return += float(reward)
        episode_length += 1
        if step >= cfg.learning_starts and step % cfg.train_frequency == 0:
            indices, batch, is_weights = sample(
                buffer, sample_key, cfg.batch_size, cfg.prioritized,
                jnp.float32(cfg.per_alpha), jnp.float32(beta(step, cfg))
            )
            params, opt_state, stats, errors = update_fn(
                params, target, opt_state, batch, is_weights
            )
            if cfg.prioritized:
                buffer = update_priorities(buffer, indices, errors, cfg.per_eps)
            loss, td_error, q_mean, y_mean = map(float, stats)
            updates.append(dict(step=step, loss=loss, abs_td_error=td_error,
                                q_mean=q_mean, target_mean=y_mean))
        if step % cfg.target_update_frequency == 0:
            target = params
        if bool(done):
            episodes.append(dict(step=step, episode_return=episode_return,
                                 episode_length=episode_length))
            obs, state = env.reset(reset_key, env_params)
            episode_return, episode_length = 0., 0
        else:
            obs, state = next_obs, next_state
        if step % cfg.eval_every == 0 or step == cfg.total_steps:
            scores = np.asarray(evaluator(params, eval_keys, jnp.bool_(True)))
            evaluations.append(dict(step=step, mean=scores.mean(),
                                    median=np.median(scores),
                                    std=scores.std(ddof=1), epsilon=eps))
            if verbose:
                print(f"step={step:6d} eps={eps:.3f} "
                      f"eval mean={scores.mean():6.1f} "
                      f"episodes={len(episodes)}")
    return params, net, {
        "episodes": pd.DataFrame(episodes),
        "updates": pd.DataFrame(updates),
        "evaluation": pd.DataFrame(evaluations)
    }

smoke = replace(CFG, total_steps=24, learning_starts=4, train_frequency=2,
                target_update_frequency=8, eval_every=24, eval_episodes=2,
                buffer_capacity=32, batch_size=8)
sp, snet, slog = train(smoke, verbose=False)
assert not slog["updates"].empty
assert np.isfinite(slog["updates"]["loss"]).all()
print("Complete PER + Double + Dueling smoke test passed.")


Complete PER + Double + Dueling smoke test passed.


## 7. Train baseline DQN and combined improved DQN

In [ ]:
BASELINE = replace(CFG, double=False, dueling=False, prioritized=False)
IMPROVED = CFG
runs = {}
for label, config in [("DQN", BASELINE), ("Double + Dueling + PER", IMPROVED)]:
    print("\nTraining:", label)
    runs[label] = train(config)



Training: DQN
step=  5000 eps=0.864 eval mean= 211.3 episodes=200
step= 10000 eps=0.729 eval mean= 127.4 episodes=330
step= 15000 eps=0.593 eval mean= 178.2 episodes=445
step= 20000 eps=0.457 eval mean= 115.3 episodes=551
step= 25000 eps=0.321 eval mean=  95.0 episodes=678
step= 30000 eps=0.186 eval mean= 101.2 episodes=748
step= 35000 eps=0.050 eval mean=  89.8 episodes=822
step= 40000 eps=0.050 eval mean=  92.3 episodes=885
step= 45000 eps=0.050 eval mean=  77.9 episodes=945
step= 50000 eps=0.050 eval mean=  87.5 episodes=1007
step= 55000 eps=0.050 eval mean=  73.7 episodes=1067
step= 60000 eps=0.050 eval mean=  92.3 episodes=1153
step= 65000 eps=0.050 eval mean= 248.5 episodes=1212
step= 70000 eps=0.050 eval mean= 162.9 episodes=1275
step= 75000 eps=0.050 eval mean= 152.7 episodes=1323
step= 80000 eps=0.050 eval mean= 187.1 episodes=1340
step= 85000 eps=0.050 eval mean= 356.9 episodes=1346
step= 90000 eps=0.050 eval mean= 500.0 episodes=1360
step= 95000 eps=0.050 eval mean= 182.1 e

## 8. Learning curves and TD diagnostics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for label, (_, _, logs) in runs.items():
    ev, upd = logs["evaluation"], logs["updates"]
    axes[0].plot(ev["step"], ev["mean"], marker="o", label=label)
    axes[1].plot(upd["step"], upd["loss"].rolling(100, min_periods=1).mean(),
                 label=label)
    axes[2].plot(upd["step"], upd["abs_td_error"].rolling(100, min_periods=1).mean(),
                 label=label)
for ax, title in zip(axes, ["Greedy evaluation return", "Weighted Huber TD loss",
                            "Mean absolute TD error"]):
    ax.set(title=title, xlabel="Environment steps")
    ax.grid(alpha=.25)
    ax.legend()
axes[0].set_ylabel("Mean episode return")
plt.tight_layout()
plt.show()


## 9. Final paired evaluation and confidence intervals

In [ ]:
common_keys = jax.random.split(jax.random.PRNGKey(777), 100)
rows = []
final_scores = {}
for label, (params, net, _) in runs.items():
    evaluator = make_evaluator(net, CFG.max_episode_steps)
    for mode, greedy in [("greedy", True), ("epsilon-greedy", False)]:
        scores = np.asarray(evaluator(params, common_keys, jnp.bool_(greedy)))
        final_scores[(label, mode)] = scores
        lo, hi = bootstrap_ci(scores)
        rows.append(dict(agent=label, mode=mode, mean=scores.mean(),
                         median=np.median(scores), std=scores.std(ddof=1),
                         ci_low=lo, ci_high=hi))
display(pd.DataFrame(rows).round(2))
fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot([final_scores[(label, "greedy")] for label in runs],
           tick_labels=list(runs), showmeans=True)
ax.set(ylabel="Episode return", title="Greedy policies on common evaluation keys")
ax.grid(axis="y", alpha=.25)
plt.tight_layout()
plt.show()


## 10. Factorial ablations (optional)

The full $2^3$ design isolates main effects and interactions among Double DQN, Dueling DQN, and PER. All configurations share the same training seed, step budget, and evaluation keys. **Multiple training seeds** are still needed for reliable conclusions. Set `RUN_FACTORIAL=True` to execute all eight combinations.


In [ ]:
RUN_FACTORIAL = False
if RUN_FACTORIAL:
    from itertools import product
    factorial = []
    factorial_curves = {}
    keys = jax.random.split(jax.random.PRNGKey(5555), 100)
    for double, dueling, prioritized in product((False, True), repeat=3):
        cfg = replace(CFG, double=double, dueling=dueling, prioritized=prioritized)
        label = f"Double={int(double)} Dueling={int(dueling)} PER={int(prioritized)}"
        print("\n", label)
        p, net, logs = train(cfg, verbose=False)
        scores = np.asarray(make_evaluator(net, cfg.max_episode_steps)(
            p, keys, jnp.bool_(True)
        ))
        factorial.append(dict(double=double, dueling=dueling,
                              prioritized=prioritized, mean=scores.mean(),
                              median=np.median(scores), std=scores.std(ddof=1)))
        factorial_curves[label] = logs["evaluation"]
    factorial_df = pd.DataFrame(factorial)
    display(factorial_df.sort_values("mean", ascending=False).round(2))
    fig, ax = plt.subplots(figsize=(10, 5))
    for label, curve in factorial_curves.items():
        ax.plot(curve["step"], curve["mean"], label=label)
    ax.set(xlabel="Environment steps", ylabel="Greedy return",
           title="Factorial DQN ablation")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(alpha=.25)
    plt.show()


## 11. Multi-seed experiment (optional)

In [ ]:
RUN_MULTISEED = False
SEEDS = (0, 1, 2, 3, 4)
if RUN_MULTISEED:
    results = []
    keys = jax.random.split(jax.random.PRNGKey(4242), 100)
    for seed in SEEDS:
        for label, config in [("DQN", BASELINE), ("Improved", IMPROVED)]:
            p, net, _ = train(config, seed=seed, verbose=False)
            scores = np.asarray(make_evaluator(net, config.max_episode_steps)(
                p, keys, jnp.bool_(True)
            ))
            results.append(dict(seed=seed, agent=label,
                                mean_return=scores.mean(),
                                median_return=np.median(scores)))
    multiseed = pd.DataFrame(results)
    display(multiseed.pivot(index="seed", columns="agent", values="mean_return"))
    display(multiseed.groupby("agent")["mean_return"].agg(["mean", "std"]))


## 12. What the experiments test

- **Double DQN:** Does decoupling selection and evaluation reduce overestimation and improve returns?
- **Dueling:** Does explicitly separating $V(s)$ from centered $A(s,a)$ help where actions have similar values?
- **PER:** Does focusing updates on high-error transitions improve sample efficiency? Does it introduce instability?
- **Interactions:** Do improvements combine additively, or does one become redundant with another?

**Limitations:** A single seed is illustrative, not evidence of superiority. CartPole may be too easy to reveal consistent gains from all three extensions. PER here has linear-time probability construction, rather than a production sum-tree implementation. The TD loss is not directly comparable between uniform and importance-weighted replay.

**Next notebook:** 07 — CNN-based DQN on MinAtar.
